First of all, import the modules needed.

In [ ]:
import zipfile
import requests
import os
import optuna
import time
import shutil

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datetime import date, timedelta
from numba import njit
from pathlib import Path

Then, download and save the data.

In [ ]:
symbol = "TRUMPUSDT"
interval = "1m"
base_url = "https://data.binance.vision/data/spot/daily/klines"

start = date(2023, 1, 1)   # inclusive
end = date(2026, 2, 28)   # inclusive

download_dir = "tmp_data"
os.makedirs(download_dir, exist_ok=True)

# 1) Download and unzip daily files
d = start
while d <= end:
    ds = d.strftime("%Y-%m-%d")
    zip_name = f"{symbol}-{interval}-{ds}.zip"
    url = f"{base_url}/{symbol}/{interval}/{zip_name}"
    zip_path = os.path.join(download_dir, zip_name)

    print(f"Fetching {url}")
    r = requests.get(url)
    if r.status_code == 200:
        with open(zip_path, "wb") as f:
            f.write(r.content)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(download_dir)
        os.remove(zip_path)  # keep only CSV
        print(f"OK {ds}")
    else:
        print(f"Missing or error for {ds}: {r.status_code}")
    d += timedelta(days=1)

# 2) Concatenate all CSVs in time order
csv_files = sorted(f for f in os.listdir(download_dir) if f.endswith(".csv") and f.startswith(f"{symbol}-{interval}-"))

dfs = []
for fname in csv_files:
    path = os.path.join(download_dir, fname)
    df = pd.read_csv(path, header=None)
    # Binance kline schema:
    # 0 open time, 1 open, 2 high, 3 low, 4 close, 5 volume,
    # 6 close time, 7 quote asset volume, 8 number of trades,
    # 9 taker buy base volume, 10 taker buy quote volume, 11 ignore
    dfs.append(df)

if not dfs:
    raise RuntimeError("No CSVs downloaded; check dates and URLs.")

full = pd.concat(dfs, ignore_index=True)

full = full.sort_values(0).drop_duplicates(subset=0)
full = full.iloc[:, :-1]  # Drop last column (useless)
threshold = 1e14
# Convert all time data in time column to ms
full.iloc[:, 0] = np.where(full.iloc[:, 0] > threshold, full.iloc[:, 0] // 1000, full.iloc[:, 0])
full.iloc[:, 6] = np.where(full.iloc[:, 6] > threshold, full.iloc[:, 6] // 1000, full.iloc[:, 6])

# 3) Save a single merged CSV
save_dir = "data"
os.makedirs(save_dir, exist_ok=True)
out_file = f"{save_dir}/{symbol}-{interval}-{start.strftime('%Y-%m-%d')}_{end.strftime('%Y-%m-%d')}.csv"
full.to_csv(out_file, index=False, header=["OpenTimestamp[ms]", "Open", "High", "Low", "Close", "Volume", "CloseTimestamp[ms]", "QuoteAssetVolume", "NumberOfTrades", "TakerBuyBaseVolume", "TakerBuyQuoteVolume"])
print("Merged CSV written to:", out_file)

# Clear all the temp .zip files
root = Path(download_dir)
for item in root.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

#Define the backtest functions:

In [ ]:
# ====================== JITTED BACKTEST CORE ======================
@njit(fastmath=True, cache=True)
def simulate(closes: np.ndarray, entry_signals: np.ndarray, macd_cross_downs: np.ndarray, sl_rate: float, tp_rate: float, max_hold: int, initial_capital: float, fee_rate: float):
    n = len(closes)
    equity = np.empty(n, dtype=np.float64)
    cash = initial_capital
    position = 0.0
    entry_price = 0.0
    planned_exit = -1
    total_traded = 0.0
    max_lev = 0.0
    trades = 0

    for i in range(n):
        close_p = closes[i]

        # EXIT FIRST
        if position > 0.0:
            hit_sl = close_p <= entry_price * (1.0 - sl_rate)
            hit_tp = close_p >= entry_price * (1.0 + tp_rate)
            hit_macd = macd_cross_downs[i]
            hit_time = (i >= planned_exit)

            if hit_sl or hit_tp or hit_macd or hit_time:
                sell_notional = position * close_p
                total_traded += sell_notional
                cash += sell_notional * (1.0 - fee_rate) if fee_rate > 0.0 else sell_notional
                position = 0.0
                entry_price = 0.0
                planned_exit = -1

        # ENTRY
        if (position == 0.0 and entry_signals[i] and cash > 0.01 and (i + max_hold < n)):
            notional = cash / (1.0 + fee_rate) if fee_rate > 0.0 else cash
            size = notional / close_p
            position = size
            cash = 0.0
            total_traded += notional
            entry_price = close_p
            planned_exit = i + max_hold
            trades += 1

        equity[i] = cash + position * close_p

        if equity[i] > 0.0 and position > 0.0:
            lev = (position * close_p) / equity[i]
            if lev > max_lev:
                max_lev = lev

    return equity, total_traded, max_lev, trades


def compute_metrics(equity: np.ndarray, initial_capital: float, total_traded: float, n: int):
    final_pnl = equity[-1] - initial_capital
    if n < 2:
        return {'final_pnl': final_pnl, 'sharpe': 0.0, 'ann_sharpe': 0.0,
                'ann_turnover': 0.0, 'max_dd_pct': 0.0}

    returns = np.diff(equity) / equity[:-1]
    sharpe = 0.0
    ann_sharpe = 0.0
    if len(returns) > 30 and np.std(returns) > 1e-8:
        sharpe = np.mean(returns) / np.std(returns)
        annual_factor = 365.25 * 24 * 60
        ann_sharpe = sharpe * np.sqrt(annual_factor)

    cummax = np.maximum.accumulate(equity)
    dd = (equity - cummax) / cummax
    max_dd_pct = -dd.min() * 100 if dd.min() < 0 else 0.0

    years = n / (365.25 * 24 * 60)
    ann_turnover = (total_traded / initial_capital) / years if years > 0 else 0.0

    return {
        'final_pnl': final_pnl,
        'sharpe': sharpe,
        'ann_sharpe': ann_sharpe,
        'ann_turnover': ann_turnover,
        'max_dd_pct': max_dd_pct
    }


# ====================== SIGNAL COMPUTATION HELPER ======================
def compute_entry_exit_signals(closes, vwap, lows, ema_fast=12, ema_slow=26, signal_len=9):
    close_s = pd.Series(closes)
    ema_f = close_s.ewm(span=ema_fast, adjust=False).mean().values
    ema_s = close_s.ewm(span=ema_slow, adjust=False).mean().values
    macd_line = ema_f - ema_s
    sig_line = pd.Series(macd_line).ewm(span=signal_len, adjust=False).mean().values

    cross_up = np.full(len(closes), False)
    cross_up[1:] = (macd_line[:-1] <= sig_line[:-1]) & (macd_line[1:] > sig_line[1:])

    bounce_support = (lows <= vwap) & (closes > vwap)
    entry_signals = (closes > vwap) & cross_up & bounce_support

    cross_down = np.full(len(closes), False)
    cross_down[1:] = (macd_line[:-1] >= sig_line[:-1]) & (macd_line[1:] < sig_line[1:])

    return entry_signals, cross_down

Then, run the backtest (specifically for $(x, y) = (10, 3)$):

In [ ]:
csv_path = "data/TRXUSDT-1m-2023-01-01_2026-02-28.csv"
sl, tp, max_hold = 0.002, 0.0048, 113
short_ema, long_ema, signal = 17, 26, 28
print("Loading CSV...")
df = pd.read_csv(csv_path)
df['datetime'] = pd.to_datetime(df["OpenTimestamp[ms]"], unit="ms")
df = df.sort_values("datetime").reset_index(drop=True)
n = len(df)
print(f"Data loaded: {n:,} minutes (~{n / (60 * 24):.1f} days)")

# Pre-compute only VWAP (fixed)
df['date'] = df['datetime'].dt.date
df['typical_price'] = (df['High'] + df['Low'] + df['Close']) / 3
df['VWAP'] = (df['typical_price'] * df['Volume']).groupby(df['date']).cumsum() / df['Volume'].groupby(df['date']).cumsum()

closes = df['Close'].values
vwap = df['VWAP'].values
lows = df['Low'].values
initial_capital = 1_000_000.0

# ====================== DEFAULT BACKTEST ======================
print(f"\nRunning default...")
entry_def, cross_def = compute_entry_exit_signals(closes, vwap, lows, short_ema, long_ema, signal)

equity_before, traded_before, _, trades_before = simulate(closes, entry_def, cross_def, sl, tp, max_hold, initial_capital, 0.0)
metrics_before = compute_metrics(equity_before, initial_capital, traded_before, n)
equity_after, traded_after, max_lev, trades_after = simulate(closes, entry_def, cross_def, sl, tp, max_hold, initial_capital, 0.001)
metrics_after = compute_metrics(equity_after, initial_capital, traded_after, n)

print("\n" + "=" * 90)
print("DEFAULT RESULTS")
print("=" * 90)
print(f"Final PnL BEFORE    : ${metrics_before['final_pnl']:,.2f}")
print(f"Final PnL AFTER     : ${metrics_after['final_pnl']:,.2f}")
print(f"Ann. Sharpe BEFORE  : {metrics_before['ann_sharpe']:.2f}")
print(f"Ann. Sharpe AFTER   : {metrics_after['ann_sharpe']:.2f}")
print(f"Max DD BEFORE       : {metrics_before['max_dd_pct']:.2f}%")
print(f"Max DD AFTER        : {metrics_after['max_dd_pct']:.2f}%")
print(f"Total trades BEFORE : {trades_before:,}")
print(f"Total trades AFTER  : {trades_after:,}")
print(f"Ann. Turnover       : {metrics_after['ann_turnover']:.2f}x")
print(f"Max Leverage        : {max_lev:.2f}x")
print("=" * 90)

# Save equity curve for Flask dashboard
equity_df = pd.DataFrame({'datetime': df['datetime'], 'pnl_before': equity_before - initial_capital, 'pnl_after': equity_after - initial_capital})
equity_df.to_csv('equity_curve.csv', index=False)
print("Saved 'equity_curve.csv' for the Flask dashboard.")

# Cumulative PnL plot (static)
plt.figure(figsize=(14, 7))
plt.plot(df['datetime'], equity_before - initial_capital, label='Before 0.1% cost', color='royalblue')
plt.plot(df['datetime'], equity_after - initial_capital, label='After 0.1% cost', color='crimson')
plt.title(f'Mean-Reversion Backtest (x={max_hold}, SL={sl * 100:.2f}, TP={tp * 100:.2f}%)', fontsize=16)
plt.xlabel('Time')
plt.ylabel('Cumulative PnL (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Now, we use `optuna` to optimize...

In [ ]:
# ====================== OPTUNA OBJECTIVE ======================
def objective(trial, closes, vwap, lows, initial_capital, n, fee_rate=0.001):
    ema_fast = trial.suggest_int('ema_fast', 5, 30)
    ema_slow = trial.suggest_int('ema_slow', 20, 60)
    signal_len = trial.suggest_int('signal_len', 5, 30)
    x = trial.suggest_int('x', 1, 120)          # max hold minutes
    sl = trial.suggest_float('sl', 0.001, 0.01)
    tp = trial.suggest_float('tp', 0.001, 0.05)

    entry_signals, macd_cross_downs = compute_entry_exit_signals(
        closes, vwap, lows, ema_fast, ema_slow, signal_len)

    equity, traded, _, _ = simulate(closes, entry_signals, macd_cross_downs,
                                    sl, tp, x, initial_capital, fee_rate)
    metrics = compute_metrics(equity, initial_capital, traded, n)
    return metrics['ann_sharpe']


# ====================== OPTUNA (6 params) ======================
n_trials = 1_000

print(f"\nStarting Optuna ({n_trials:,} trials) — optimizing EMA periods + x + SL + TP...")
start_time = time.time()

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=67))
study.optimize(lambda trial: objective(trial, closes, vwap, lows, initial_capital, n, fee_rate=0.0),
               n_trials=n_trials, show_progress_bar=True)

best = study.best_params
best_sharpe = study.best_value
print(f"\nOPTUNA BEST: EMA_fast={best['ema_fast']}, EMA_slow={best['ema_slow']}, "
      f"Signal={best['signal_len']}, x={best['x']}, SL={best['sl'] * 100:.2f}%, "
      f"TP={best['tp'] * 100:.2f}% → Sharpe = {best_sharpe:.4f}")
print(f"Finished in {time.time() - start_time:.1f}s")

# ====================== RUN BEST & SAVE ======================
print("\nRunning best parameters...")
entry_best, cross_best = compute_entry_exit_signals(closes, vwap, lows, best['ema_fast'], best['ema_slow'], best['signal_len'])

equity_before_best, _, _, _ = simulate(closes, entry_best, cross_best, best['sl'], best['tp'], best['x'], initial_capital, 0.0)
equity_after_best, traded_best, max_lev_best, trades_best = simulate(closes, entry_best, cross_best, best['sl'], best['tp'], best['x'], initial_capital, 0.001)
metrics_best = compute_metrics(equity_after_best, initial_capital, traded_best, n)

equity_df = pd.DataFrame({
    'datetime': df['datetime'],
    'pnl_before': equity_before_best - initial_capital,
    'pnl_after': equity_after_best - initial_capital
})
equity_df.to_csv('equity_curve.csv', index=False)
print("✅ Saved 'equity_curve.csv' (BEST parameters)")

print("\n" + "=" * 90)
print(f"BEST RESULTS (EMA{best['ema_fast']}/{best['ema_slow']}/{best['signal_len']}, x={best['x']}, SL={best['sl'] * 100:.2f}%, TP={best['tp'] * 100:.2f}%)")
print("=" * 90)
print(f"Final PnL AFTER  : ${metrics_best['final_pnl']:,.2f}")
print(f"Sharpe AFTER     : {metrics_best['sharpe']:.4f}")
print(f"Ann. Sharpe      : {metrics_best['ann_sharpe']:.2f}")
print(f"Max DD           : {metrics_best['max_dd_pct']:.2f}%")
print(f"Total trades     : {trades_best:,}")
print(f"Ann. Turnover    : {metrics_best['ann_turnover']:.2f}x")
print(f"Max Leverage     : {max_lev_best:.2f}x")
print("=" * 90)

plt.figure(figsize=(14, 7))
plt.plot(df['datetime'], equity_before_best - initial_capital, label='Before cost', color='royalblue')
plt.plot(df['datetime'], equity_after_best - initial_capital, label='After cost', color='crimson')
plt.title(f'BEST Strategy (EMA{best['ema_fast']}/{best['ema_slow']}/{best['signal_len']}, x={best['x']}, SL={best['sl'] * 100:.2f}%, TP={best['tp'] * 100:.2f}%)', fontsize=16)
plt.xlabel('Time')
plt.ylabel('Cumulative PnL (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()